In [1]:
import torch
import os
os.chdir('../../')

In [2]:
from scipy import linalg
import numpy as np
import os, torch
from tqdm import tqdm
from reports.util import load_config


@torch.no_grad()
def _trace_sqrtm_product(C1: torch.Tensor, C2: torch.Tensor) -> torch.Tensor:
    # Tr sqrtm(C1 @ C2) = Tr sqrt( C1^{1/2} C2 C1^{1/2} )
    s, U = torch.linalg.eigh(C1)                 # C1 = U diag(s) U^T
    s = s.clamp_min(0)
    C1h = (U * s.sqrt()) @ U.t()                 # C1^{1/2}
    M   = C1h @ C2 @ C1h
    w   = torch.linalg.eigvalsh((M + M.t()) * 0.5).clamp_min(0)
    return w.sqrt().sum()

@torch.no_grad()
def calc_fid_stats(mu1, sigma1, mu2, sigma2, eps: float = 1e-6) -> float:
    # 모두 float64 + 동일 device로 정렬
    C1 = torch.as_tensor(sigma1, dtype=torch.float64)
    device = C1.device
    C2 = torch.as_tensor(sigma2, dtype=torch.float64).to(device)
    m1 = torch.as_tensor(mu1,    dtype=torch.float64).to(device).flatten()
    m2 = torch.as_tensor(mu2,    dtype=torch.float64).to(device).flatten()

    D = m1.numel()
    I = torch.eye(D, dtype=torch.float64, device=device)

    # 대칭화 + 정칙화
    C1 = (C1 + C1.t()) * 0.5 + eps * I
    C2 = (C2 + C2.t()) * 0.5 + eps * I

    diff = m1 - m2
    tr_covmean = _trace_sqrtm_product(C1, C2)
    fid = diff.dot(diff) + torch.trace(C1) + torch.trace(C2) - 2.0 * tr_covmean
    return float(fid)

@torch.no_grad()
def calc_fid_pt_dir(pt_dir: str, mu, sigma, eps: float = 1e-6, num=100000, key="inception_feature") -> float:
    # pt_dir에서 'inception_feature'를 모아서 mu1, sigma1 추정 후 FID 계산
    X = []
    for f in tqdm(os.listdir(pt_dir)[:num]):
        if f.endswith(".pt"):
            v = torch.load(os.path.join(pt_dir, f), map_location="cpu").get(key)
            if v is not None:
                X.append(torch.as_tensor(v, dtype=torch.float64).flatten())
    if len(X) < 2:
        raise ValueError("need >=2 features")

    X   = torch.stack(X, 0)                 # [N, D]
    mu1 = X.mean(0)
    Xc  = X - mu1
    sigma1 = (Xc.t() @ Xc) / (X.shape[0] - 1)  # 불편추정

    return calc_fid_stats(mu1, sigma1, mu, sigma, eps=eps)
    #return calculate_frechet_distance(mu1, sigma1, mu, sigma, eps=eps)

def calculate_frechet_distance(mu1, sigma1, mu2, sigma2, eps=1e-6):
    """Numpy implementation of the Frechet Distance.
    The Frechet distance between two multivariate Gaussians X_1 ~ N(mu_1, C_1)
    and X_2 ~ N(mu_2, C_2) is
            d^2 = ||mu_1 - mu_2||^2 + Tr(C_1 + C_2 - 2*sqrt(C_1*C_2)).

    Stable version by Dougal J. Sutherland.

    Params:
    -- mu1   : Numpy array containing the activations of a layer of the
               inception net (like returned by the function 'get_predictions')
               for generated samples.
    -- mu2   : The sample mean over activations, precalculated on an
               representative data set.
    -- sigma1: The covariance matrix over activations for generated samples.
    -- sigma2: The covariance matrix over activations, precalculated on an
               representative data set.

    Returns:
    --   : The Frechet Distance.
    """

    mu1 = np.atleast_1d(mu1)
    mu2 = np.atleast_1d(mu2)

    sigma1 = np.atleast_2d(sigma1)
    sigma2 = np.atleast_2d(sigma2)

    assert mu1.shape == mu2.shape, \
        'Training and test mean vectors have different lengths'
    assert sigma1.shape == sigma2.shape, \
        'Training and test covariances have different dimensions'

    diff = mu1 - mu2

    # Product might be almost singular
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    if not np.isfinite(covmean).all():
        msg = ('fid calculation produces singular product; '
               'adding %s to diagonal of cov estimates') % eps
        print(msg)
        offset = np.eye(sigma1.shape[0]) * eps
        covmean = linalg.sqrtm((sigma1 + offset).dot(sigma2 + offset))

    # Numerical error might give slight imaginary component
    if np.iscomplexobj(covmean):
        if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
            m = np.max(np.abs(covmean.imag))
            raise ValueError('Imaginary component {}'.format(m))
        covmean = covmean.real

    tr_covmean = np.trace(covmean)

    return (diff.dot(diff) + np.trace(sigma1)
            + np.trace(sigma2) - 2 * tr_covmean)

def get_clip_score(pt_file, key):
    data = torch.load(pt_file)
    return float(data[key])

from pathlib import Path
import numpy as np
from tqdm import tqdm

def get_clip_scores(dir):
    scores = {}
    
    for pt_file in tqdm(Path(dir).rglob('*.pt')):
        data = torch.load(pt_file)
        for key in data.keys():
            if key.startswith('clip_score'):
                clip_score = get_clip_score(pt_file, key)
                if key in scores:
                    scores[key].append(clip_score)
                else:
                    scores[key] = [clip_score]
    for key in scores.keys():
        scores[key] = np.mean(scores[key])
    return scores
        

In [ ]:
pt_dirs = [
        #'samplings/PixArt-Alpha/3.5/9/Euler/30000/euler_0',
        #'samplings/PixArt-Alpha/3.5/8/Euler/30000/euler_0',
        #'samplings/PixArt-Alpha/3.5/7/Euler/30000/euler_0',
        #'samplings/PixArt-Alpha/3.5/6/Euler/30000/euler_0',
        #'samplings/PixArt-Alpha/3.5/5/Euler/30000/euler_0',
        #'samplings/PixArt-Alpha/3.5/4/Euler/30000/euler_0',
        'samplings/PixArt-Alpha/3.5/3/Euler/30000/euler_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    print(pt_dir)

    # fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    # print('FID :', fid)
    
    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

samplings/PixArt-Alpha/3.5/3/Euler/30000/euler_0


30000it [01:48, 276.56it/s]

clip_score_ViT-L/14 0.2275
clip_score_ViT-L/14@336px 0.2339
clip_score_RN101 0.4469


In [3]:
pt_dirs = [
        'samplings/PixArt-Alpha/3.5/9/DPM-Solver/30000/dpm_0',
        'samplings/PixArt-Alpha/3.5/8/DPM-Solver/30000/dpm_0',
        'samplings/PixArt-Alpha/3.5/7/DPM-Solver/30000/dpm_0',
        'samplings/PixArt-Alpha/3.5/6/DPM-Solver/30000/dpm_0',
        'samplings/PixArt-Alpha/3.5/5/DPM-Solver/30000/dpm_0',
        'samplings/PixArt-Alpha/3.5/4/DPM-Solver/30000/dpm_0',
        'samplings/PixArt-Alpha/3.5/3/DPM-Solver/30000/dpm_0',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    print(pt_dir)

    # fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    # print('FID :', fid)
    
    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

    print()

samplings/PixArt-Alpha/3.5/9/DPM-Solver/30000/dpm_0


0it [00:00, ?it/s]

30000it [00:57, 524.79it/s]


clip_score_ViT-L/14 0.2586
clip_score_ViT-L/14@336px 0.2667
clip_score_RN101 0.4771
samplings/PixArt-Alpha/3.5/8/DPM-Solver/30000/dpm_0


30000it [00:53, 561.70it/s]


...

clip_score_ViT-L/14 0.2562
clip_score_ViT-L/14@336px 0.2640
clip_score_RN101 0.4746
samplings/PixArt-Alpha/3.5/4/DPM-Solver/30000/dpm_0


30000it [00:48, 617.70it/s]


clip_score_ViT-L/14 0.2497
clip_score_ViT-L/14@336px 0.2570
clip_score_RN101 0.4676
samplings/PixArt-Alpha/3.5/3/DPM-Solver/30000/dpm_0


30000it [00:48, 621.79it/s]

clip_score_ViT-L/14 0.2229
clip_score_ViT-L/14@336px 0.2302
clip_score_RN101 0.4422


In [8]:
pt_dirs = [
        #'samplings/PixArt-Alpha/3.5/9/Dual-Solver/30000/rn_0/',
        #'samplings/PixArt-Alpha/3.5/8/Dual-Solver/30000/rn_0/',
        #'samplings/PixArt-Alpha/3.5/7/Dual-Solver/30000/rn_0/',
        #'samplings/PixArt-Alpha/3.5/6/Dual-Solver/30000/rn_0/',
        #'samplings/PixArt-Alpha/3.5/5/Dual-Solver/30000/rn_0/',
        #'samplings/PixArt-Alpha/3.5/4/Dual-Solver/30000/rn_0/',
        'samplings/PixArt-Alpha/3.5/3/Dual-Solver/30000/rn_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    print(pt_dir)

    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print('FID :', fid)
    
    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

    print()

samplings/PixArt-Alpha/3.5/3/Dual-Solver/30000/rn_0/


100%|██████████| 30001/30001 [00:30<00:00, 978.73it/s] 


FID : 79.56247649687157


30000it [02:01, 245.96it/s]

clip_score_ViT-B/16 0.2822
clip_score_ViT-L/14 0.2359
clip_score_ViT-L/14@336px 0.2410
clip_score_RN101 0.4499



In [4]:
pt_dirs = [
        #'samplings/PixArt-Alpha/3.5/9/Dual-Solver/30000/traj2_0/',
        #'samplings/PixArt-Alpha/3.5/8/Dual-Solver/30000/traj2_0/',
        'samplings/PixArt-Alpha/3.5/7/Dual-Solver/30000/traj2_0/',
        #'samplings/PixArt-Alpha/3.5/6/Dual-Solver/30000/traj2_0/',
        #'samplings/PixArt-Alpha/3.5/5/Dual-Solver/30000/traj2_0/',
        #'samplings/PixArt-Alpha/3.5/4/Dual-Solver/30000/traj2_0/',
        #'samplings/PixArt-Alpha/3.5/3/Dual-Solver/30000/traj2_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    print(pt_dir)

    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print('FID :', fid)
    
    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

    print()

samplings/PixArt-Alpha/3.5/7/Dual-Solver/30000/traj2_0/


100%|██████████| 30001/30001 [00:19<00:00, 1515.22it/s]


FID : 25.009475261246223


30000it [01:10, 425.48it/s]

clip_score_ViT-L/14 0.2530
clip_score_ViT-L/14@336px 0.2631
clip_score_RN101 0.4754



In [3]:
pt_dirs = [
        'samplings/PixArt-Alpha/3.5/9/DS-Solver_DDPM/30000/ds_0/',
        'samplings/PixArt-Alpha/3.5/8/DS-Solver_DDPM/30000/ds_0/',
        'samplings/PixArt-Alpha/3.5/7/DS-Solver_DDPM/30000/ds_0/',
        'samplings/PixArt-Alpha/3.5/6/DS-Solver_DDPM/30000/ds_0/',
        'samplings/PixArt-Alpha/3.5/5/DS-Solver_DDPM/30000/ds_0/',
        'samplings/PixArt-Alpha/3.5/4/DS-Solver_DDPM/30000/ds_0/',
        'samplings/PixArt-Alpha/3.5/3/DS-Solver_DDPM/30000/ds_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    print(pt_dir)

    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print('FID :', fid)
    
    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

    print()

samplings/PixArt-Alpha/3.5/9/DS-Solver_DDPM/30000/ds_0/


100%|██████████| 30001/30001 [00:18<00:00, 1651.04it/s]


FID : 25.22578308050629


30000it [01:16, 393.20it/s]


clip_score_ViT-B/16 0.3088
clip_score_ViT-L/14 0.2572
clip_score_ViT-L/14@336px 0.2651
clip_score_RN101 0.4764

samplings/PixArt-Alpha/3.5/8/DS-Solver_DDPM/30000/ds_0/


...

clip_score_ViT-B/16 0.2935
clip_score_ViT-L/14 0.2445
clip_score_ViT-L/14@336px 0.2491
clip_score_RN101 0.4568

samplings/PixArt-Alpha/3.5/3/DS-Solver_DDPM/30000/ds_0/


100%|██████████| 30001/30001 [00:15<00:00, 1890.67it/s]


FID : 118.0143692588133


30000it [01:10, 423.81it/s]

clip_score_ViT-B/16 0.2630
clip_score_ViT-L/14 0.2154
clip_score_ViT-L/14@336px 0.2188
clip_score_RN101 0.4303



In [4]:
pt_dirs = [
        'samplings/PixArt-Alpha/3.5/9/BNS-Solver_Sep/30000/bns_0/',
        'samplings/PixArt-Alpha/3.5/8/BNS-Solver_Sep/30000/bns_0/',
        'samplings/PixArt-Alpha/3.5/7/BNS-Solver_Sep/30000/bns_0/',
        'samplings/PixArt-Alpha/3.5/6/BNS-Solver_Sep/30000/bns_0/',
        'samplings/PixArt-Alpha/3.5/5/BNS-Solver_Sep/30000/bns_0/',
        'samplings/PixArt-Alpha/3.5/4/BNS-Solver_Sep/30000/bns_0/',
        'samplings/PixArt-Alpha/3.5/3/BNS-Solver_Sep/30000/bns_0/',
        ]
for pt_dir in pt_dirs:
    if not os.path.exists(pt_dir):
        continue
    data = torch.load('mscoco2014_fid/coco2014_val_30k_fid_stats.pt')
    print(pt_dir)

    fid = calc_fid_pt_dir(pt_dir, data['mu'], data['sigma'])
    print('FID :', fid)
    
    scores = get_clip_scores(pt_dir)
    for key in scores.keys():
        print(key, f"{scores[key]:.4f}")

    print()

samplings/PixArt-Alpha/3.5/9/BNS-Solver_Sep/30000/bns_0/


100%|██████████| 30001/30001 [00:16<00:00, 1835.77it/s]


FID : 24.15095420843454


30000it [01:12, 412.79it/s]


clip_score_ViT-B/16 0.3045
clip_score_ViT-L/14 0.2535
clip_score_ViT-L/14@336px 0.2640
clip_score_RN101 0.4757

samplings/PixArt-Alpha/3.5/8/BNS-Solver_Sep/30000/bns_0/


...

clip_score_ViT-B/16 0.2923
clip_score_ViT-L/14 0.2392
clip_score_ViT-L/14@336px 0.2495
clip_score_RN101 0.4582

samplings/PixArt-Alpha/3.5/3/BNS-Solver_Sep/30000/bns_0/


100%|██████████| 30001/30001 [00:16<00:00, 1820.54it/s]


FID : 125.65558934343738


30000it [01:15, 395.01it/s]

clip_score_ViT-B/16 0.2735
clip_score_ViT-L/14 0.2201
clip_score_ViT-L/14@336px 0.2268
clip_score_RN101 0.4320

